# 17 — ImprovedCNN1D: residual + wider pool + HuberLoss(delta=10)

Changes from SmallCNN (nb16):
1. `AdaptiveAvgPool1d(8)` -> `AdaptiveAvgPool1d(16)` — less aggressive compression
2. Block3 residual skip connection (1x1 conv shortcut, 16->32 channels)
3. `HuberLoss(delta=1.0)` -> `HuberLoss(delta=10.0)` — proper Huber for % scale

Reference: SmallCNN CV RMSE_le170 folds = [13.55, 12.42, 46.35, 20.34, 11.96], mean=20.92%
Target: Fold3 < 30%, Fold4 < 15%, mean < 15% to approach ET best (12.96%)

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission
from src.preprocessing import snv

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_T = 200.0

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'y range: {y.min():.1f} - {y.max():.1f}%')
print(f'y >170: {(y>170).sum()} samples ({(y>170).mean()*100:.1f}%)')

In [ ]:
# ==== ImprovedCNN1D ====
class ImprovedCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        # Block 1-2: downsampling (no residual, channels change too much)
        self.block12 = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
        )
        # Block 3: conv + BN (residual added in forward)
        self.conv3     = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32))
        self.shortcut3 = nn.Conv1d(16, 32, kernel_size=1)  # 1x1 projection shortcut

        # Wider global pooling: 8 -> 16
        self.pool = nn.AdaptiveAvgPool1d(16)

        # FC head
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(32 * 16, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        h = self.block12(x.unsqueeze(1))          # (B, 16, ~388)
        h = torch.relu(self.conv3(h) + self.shortcut3(h))  # residual
        h = self.pool(h)                          # (B, 32, 16)
        return self.fc(h.view(x.size(0), -1)).squeeze(1)

# Verify
_m = ImprovedCNN1D()
n_p = sum(p.numel() for p in _m.parameters())
print(f'Total params: {n_p:,}')

# Breakdown
for name, module in _m.named_modules():
    n = sum(p.numel() for p in module.parameters(recurse=False))
    if n > 0:
        print(f'  {name:20s}: {n:,}')

assert 10000 < n_p < 40000, f'Unexpected param count: {n_p}'
print(f'OK: {n_p:,} params (SmallCNN was 12,257)')

In [ ]:
# ==== Preprocessing: SNV -> SG1(41,3,1) ====
def preprocess(R):
    A = R.copy().astype(float)
    A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
    A = savgol_filter(A, window_length=41, polyorder=3, deriv=1, axis=1)
    return A.astype(np.float32)

X_pp    = preprocess(X_raw)
X_pp_te = preprocess(X_test_raw)
print(f'Preprocessed: {X_pp.shape}, range [{X_pp.min():.3f}, {X_pp.max():.3f}]')


# ==== Training function ====
def train_model(Xtr, ytr, Xva, yva, n_epochs=100, batch=32, lr=1e-3, patience=20,
                verbose=False):
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr).astype(np.float32)
    Xva_s = sc.transform(Xva).astype(np.float32)

    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                        batch_size=batch, shuffle=True, drop_last=False)

    model = ImprovedCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.HuberLoss(delta=10.0)  # delta=10: quadratic within +-10%, linear beyond

    best_val  = float('inf')
    best_state = None
    best_preds = None
    no_improve = 0
    loss_log   = []

    for epoch in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            pred = model(xb)
            loss = crit(pred, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item() * len(yb)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(Xva_t)
            val_loss = crit(val_pred, yva_t := torch.from_numpy(
                yva.astype(np.float32)).to(DEVICE)).item()

        # track RMSE for logging (not Huber)
        val_rmse = float(torch.sqrt(torch.mean((val_pred - yva_t)**2)).item())
        tr_rmse  = (ep_loss * 2 / len(ytr)) ** 0.5  # approximate from Huber

        loss_log.append({'epoch': epoch+1, 'train_rmse_approx': tr_rmse,
                         'val_rmse': val_rmse, 'val_huber': val_loss})

        if val_loss < best_val:
            best_val   = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_preds = val_pred.cpu().numpy()
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch + 1) % 10 == 0:
            print(f'    ep{epoch+1:3d}: val_rmse={val_rmse:.2f}')
        if no_improve >= patience:
            if verbose: print(f'    Early stop ep{epoch+1}')
            break

    model.load_state_dict(best_state)
    return model, sc, best_preds, loss_log


def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan

def rmse_all(yt, yp):
    return float(np.sqrt(np.mean((yt-yp)**2)))

print('ImprovedCNN1D + HuberLoss(delta=10) ready.')

## GroupKFold CV (health check)

Reference (SmallCNN+HuberLoss(1)): [13.55, 12.42, 46.35, 20.34, 11.96] mean=20.92%

In [ ]:
print('=== GroupKFold CV (ImprovedCNN, HuberLoss delta=10) ===')

fold_results = []
oof_y_list, oof_p_list = [], []
all_loss_logs = []

for fi, (tr, va) in enumerate(SPLITS):
    Xtr, Xva = X_pp[tr], X_pp[va]
    ytr, yva = y[tr],    y[va]

    model_f, sc_f, best_p, llog = train_model(
        Xtr, ytr, Xva, yva, verbose=True)

    r_all = rmse_all(yva, best_p)
    r_le  = rmse_le(yva,  best_p)
    stop_ep = llog[-1]['epoch']

    fold_results.append({'fold': fi+1, 'RMSE_all': round(r_all,2),
                         'RMSE_le170': round(r_le,2), 'stop_ep': stop_ep})
    oof_y_list.append(yva); oof_p_list.append(best_p)
    all_loss_logs.append(llog)

    print(f'  Fold {fi+1}: RMSE_all={r_all:.2f}  RMSE_le170={r_le:.2f}'
          f'  stop_ep={stop_ep}')

oof_y = np.concatenate(oof_y_list)
oof_p = np.concatenate(oof_p_list)

mean_all = np.mean([r['RMSE_all']   for r in fold_results])
mean_le  = np.mean([r['RMSE_le170'] for r in fold_results])

print()
print(f'  CV mean RMSE_all   : {mean_all:.2f}%')
print(f'  CV mean RMSE_le170 : {mean_le:.2f}%')
print(f'  Folds              : {[r["RMSE_le170"] for r in fold_results]}')
print()
print('OOF distribution:')
print(f'  min={oof_p.min():.1f}  mean={oof_p.mean():.1f}  '
      f'max={oof_p.max():.1f}  std={oof_p.std():.1f}')
print(f'  neg: {(oof_p<0).sum()}  >170: {(oof_p>170).sum()}')

print()
print('Comparison vs SmallCNN+Huber(1):')
ref = [13.55, 12.42, 46.35, 20.34, 11.96]
for fi, (r, prev) in enumerate(zip([r["RMSE_le170"] for r in fold_results], ref)):
    diff = r - prev
    sign = '+' if diff > 0 else ''
    print(f'  Fold {fi+1}: {r:.2f}%  (prev={prev:.2f}%, {sign}{diff:.2f})')

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for fi, llog in enumerate(all_loss_logs):
    eps  = [r['epoch']    for r in llog]
    vals = [r['val_rmse'] for r in llog]
    ax.plot(eps, vals, label=f'Fold {fi+1}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val RMSE')
ax.set_title('Val RMSE per fold (ImprovedCNN)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
mask = oof_y <= 170
ax.scatter(oof_y[mask], oof_p[mask], s=6, alpha=0.4)
lim = max(oof_y[mask].max(), oof_p[mask].max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=0.8)
ax.set_xlabel('Actual (%)'); ax.set_ylabel('Predicted (%)')
ax.set_title(f'OOF scatter (y<=170, RMSE_le170={mean_le:.2f}%)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/nb17_improvedcnn_cv.png', dpi=110)
plt.close()
print('Saved: results/nb17_improvedcnn_cv.png')

In [ ]:
print('=== Full train -> test prediction ===')

avg_stop = int(np.mean([r['stop_ep'] for r in fold_results]))
print(f'CV avg early-stop epoch: {avg_stop}')

sc_full = StandardScaler()
Xtr_s   = sc_full.fit_transform(X_pp).astype(np.float32)
Xte_s   = sc_full.transform(X_pp_te).astype(np.float32)

model_final = ImprovedCNN1D().to(DEVICE)
opt_f  = torch.optim.Adam(model_final.parameters(), lr=1e-3)
sched_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
crit_f  = nn.HuberLoss(delta=10.0)

Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
ytr_t = torch.from_numpy(y.astype(np.float32)).to(DEVICE)
loader_f = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)

print(f'Training {avg_stop} epochs on full train...')
for ep in range(avg_stop):
    model_final.train()
    for xb, yb in loader_f:
        loss = crit_f(model_final(xb), yb)
        opt_f.zero_grad(); loss.backward(); opt_f.step()
    sched_f.step()
    if (ep+1) % 20 == 0:
        model_final.eval()
        with torch.no_grad():
            tr_rmse = torch.sqrt(torch.mean((model_final(Xtr_t)-ytr_t)**2)).item()
        print(f'  ep{ep+1}: train_rmse={tr_rmse:.2f}')

model_final.eval()
with torch.no_grad():
    Xte_t   = torch.from_numpy(Xte_s).to(DEVICE)
    te_pred = model_final(Xte_t).cpu().numpy()

te_clip = np.clip(te_pred, 0, CLIP_T)
print(f'Test: min={te_clip.min():.1f}  mean={te_clip.mean():.1f}  '
      f'max={te_clip.max():.1f}  >170: {(te_clip>170).sum()}')

In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

make_submission(test_meta, te_clip, '../submissions/sub_improvedcnn.csv')
print('Saved: submissions/sub_improvedcnn.csv')

print()
print('=== Final Summary ===')
n_params = sum(p.numel() for p in ImprovedCNN1D().parameters())
print(f'Model: ImprovedCNN1D, {n_params:,} params')
print(f'Loss : HuberLoss(delta=10.0)')
print(f'Pool : AdaptiveAvgPool1d(16)  [was 8]')
print(f'Skip : residual at Block3')
print(f'Preprocessing: SNV -> SG1(41,3,1)')
print()
print(f'CV RMSE_le170 (5-fold): {[r["RMSE_le170"] for r in fold_results]}')
print(f'CV mean RMSE_le170    : {mean_le:.2f}%')
print(f'Test mean             : {te_clip.mean():.1f}% (target ~40-50%)')
print()
print('Reference:')
print('  SmallCNN+Huber(1) : [13.55,12.42,46.35,20.34,11.96] mean=20.92%')
print('  SmallCNN+MSE      : [12.98,15.36,53.20,32.58,14.76] mean=25.78%')
print('  ET best (nb12)    : CV=12.96% -> LB=21.20')
print('  ET wide (nb13)    : CV=16.23% -> LB=18.35 (current LB best)')